In [4]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

/home/ad.rapidops.com/sakshi.sharma/Desktop/SALESMATE_CHATBOT/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


Total characters: 43047


In [5]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


In [7]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = InMemoryVectorStore(embedding=embedding_model)

In [8]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['e67fd711-fab2-4b57-9629-155c95e3f9f1', 'c64d01e2-903f-45f3-8e29-867b049e46d8', 'd471e651-463a-4ce2-b35e-ef535500fd19']


In [12]:
from sentence_transformers import SentenceTransformer,util
import sentence_transformers

# loading the model

model=SentenceTransformer("all-MiniLM-L6-v2")
query = "What are the advancements in NLP?"
documents = [
    "Machine learning enables advancements in NLP.",
    "Climate change is a pressing issue globally.",
    "Natural language processing allows machines to understand text."
]




In [18]:
query_embedding=model.encode(query)
document_embedding= model.encode(documents)
len(query_embedding)

384

In [16]:
result=util.semantic_search(query_embedding,document_embedding,top_k=1)
result
matched_doc=documents[result[0][0]['corpus_id']]
matched_doc

'Machine learning enables advancements in NLP.'